# KEYWORD SEARCH vs SEMANTIC SEARCH

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio sentence-transformers scikit-learn -q

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("Imports successful!")

In [ ]:

# Sample documents (like an HR policy)
documents = [
    "Employees are entitled to 18 days of annual vacation.",
    "The company provides health insurance for all full-time staff.",
    "Sick leave can be availed with a medical certificate.",
    "Work from home is allowed 2 days per week.",
    "Annual performance reviews happen in December.",
    "Maternity leave is 6 months as per company policy.",
    ]
# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')
# Generate embeddings for all documents
doc_embeddings = model.encode(documents)
print("=" * 60)
print("DOCUMENT EMBEDDINGS")
print("=" * 60)
for i, doc in enumerate(documents):
  print(f"  D{i+1}: {doc}")
# User query
query = "How much leave can an employee take?"
print(f"\n 🔍 QUERY: '{query}'")
# ---- KEYWORD SEARCH (naive) ---
print("\n" + "=" * 60)
print(" ❌ KEYWORD SEARCH (word matching)")
print("=" * 60)
query_words = query.lower().split()
for i, doc in enumerate(documents):
  doc_lower = doc.lower()
  matches = [w for w in query_words if w in doc_lower]
  print(f"  D{i+1}: matched={matches} → score={len(matches)}")
  # ---- SEMANTIC SEARCH ---
print("\n" + "=" * 60)
print(" ✅ SEMANTIC SEARCH (meaning matching)")
print("=" * 60)
query_embedding = model.encode([query])
similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
# Rank documents
ranked = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(ranked, 1):
  print(f"  Rank {rank}: D{idx+1} (score={score:.4f})")
  print(f"{documents[idx]}")
print("\n 🏆 TOP RESULT:", documents[ranked[0][0]])

# Install libraries

In [ ]:
!pip install -q faiss-cpu sentence-transformers

#import ibraries

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

#create documents

In [ ]:
documents = [
    "Employees are entitled to 18 days of annual vacation.",
    "Health insurance is provided for all full-time staff.",
    "Sick leave requires a medical certificate.",
    "Work from home is allowed 2 days per week.",
    "Performance reviews happen in December.",
    "Maternity leave is 6 months.",
]
print("=" * 60)
print("DOCUMENT STORAGE")
print("=" * 60)
for i, document in enumerate(documents):
  print(f"Document {i}: {document}")

#Load embedding model

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded successfully!")

#Convert documents into vectors

In [ ]:
doc_embeddings = embedding_model.encode(documents)
# Convert to float32 because FAISS expects float32
doc_embeddings = doc_embeddings.astype("float32")
print("=" * 60)
print("DOCUMENT EMBEDDINGS")
print("=" * 60)
print("Number of documents :", len(documents))
print("Embedding shape     :", doc_embeddings.shape)
print("Data type           :", doc_embeddings.dtype)

#Show actual vector values

In [ ]:
print("VECTOR FOR DOCUMENT 0:")
print(doc_embeddings[0])
print("\nNumber of values:")
print(len(doc_embeddings[0]))

#Connect vectors with documents

In [ ]:
print("=" * 60)
print("DOCUMENT → VECTOR MAPPING")
print("=" * 60)
for i in range(len(documents)):
  print(f"\nDocument ID: {i}")
  print("Text       :", documents[i])
  print("Vector     :", doc_embeddings[i][:5], "...")

#Create FAISS index

In [ ]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
print("=" * 60)
print("FAISS INDEX")
print("=" * 60)
print("Vector dimension :", dimension)
print("Number of vectors:", index.ntotal)

#Store vectors inside FAISS

In [ ]:
index.add(doc_embeddings)
print("=" * 60)
print("FAISS STORAGE")
print("=" * 60)
print("Vectors stored in FAISS:", index.ntotal)

#Verify FAISS storage

In [ ]:
stored_vectors = index.reconstruct_n(0, index.ntotal)
print("Stored vector shape:", stored_vectors.shape)
print("\nFirst 5 values of vector 0:")
print(stored_vectors[0][:5])

#Create a user query

In [ ]:
query = "How many vacation days do I get?"
print("User Query:")
print(query)

#Convert query into vector

In [ ]:
query_embedding = embedding_model.encode([query])
query_embedding = query_embedding.astype("float32")
print("Query vector shape:", query_embedding.shape)
print("\nFirst 10 values:")
print(query_embedding[0][:10])

#Search FAISS

In [ ]:
k = 3
distances, indices = index.search(query_embedding, k)
print("=" * 60)
print("FAISS SEARCH")
print("=" * 60)
print("Distances:")
print(distances)
print("\nIndexes:")
print(indices)

#Understand the results

In [ ]:
print("=" * 60)
print("TOP SEARCH RESULTS")
print("=" * 60)
for rank, (distance, idx) in enumerate(
    zip(distances[0], indices[0]), 1
):
    print(f"\nRank       : {rank}")
    print(f"Document ID: {idx}")
    print(f"Distance   : {distance:.4f}")
    print(f"Document   : {documents[idx]}")

#Make the whole process visual

In [ ]:
print("""============================================================
              FAISS VECTOR SEARCH
============================================================
              1. ORIGINAL DOCUMENTS
                      ↓
              2. SENTENCE TRANSFORMER
                      ↓
              3. 384-DIMENSIONAL VECTORS
                      ↓
              4. FAISS INDEX
                      ↓
              5. STORE VECTORS
                      ↓
              6. USER ENTERS QUERY
                      ↓
              7. QUERY → 384-D VECTOR
                      ↓
              8. FAISS CALCULATES DISTANCES
                      ↓
              9. FIND CLOSEST VECTORS
                      ↓
              10. RETURN DOCUMENT IDs
                      ↓
              11. documents[ID]
                      ↓
              12. ORIGINAL RELEVANT TEXT
============================================================
         """)